# Notebook 11 — Online Distribution Classification

**Repo:** `int_serialization_benchmark-rml`  
**Layer:** `rml_extension/notebooks/`

Notebook 10 trained a learned execution selector from prior RML policy tables.

Notebook 11 moves one step earlier in the runtime pipeline:

> classify distribution regimes from streaming windows before choosing execution paths.

Constraint view:
> adaptive execution depends on detecting structure before selecting policy.

## Goals

1. Load Notebook 08 streaming windows when available.
2. Build online/window-level features:
   - entropy
   - repetition
   - locality
   - digit-length transitions
   - branch pressure
   - coherence
   - hardware pressure
3. Train interpretable classifiers for regime detection.
4. Simulate online prediction over time.
5. Connect predicted regime → policy recommendation.
6. Export CSV, JSON, Markdown report, and PNG figures.

In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

cwd = Path.cwd()
candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/int_serialization_benchmark-rml"),
    Path("/content"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "rml_extension").exists() or (c / "configs").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

RML_ROOT = REPO_ROOT / "rml_extension" if (REPO_ROOT / "rml_extension").exists() else REPO_ROOT

RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"
REPORTS_DIR = RML_ROOT / "reports"

for d in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RML_ROOT:", RML_ROOT)

## Load streaming/runtime windows

This notebook uses `notebook08_streaming_runtime_adaptation.csv` when available.

If not available, it creates a fallback streaming-window table.

In [ ]:
stream_path = RESULTS_DIR / "notebook08_streaming_runtime_adaptation.csv"

if stream_path.exists():
    df = pd.read_csv(stream_path)
    print("Loaded:", stream_path)
else:
    print("Notebook 08 output not found; creating fallback streaming-window table.")
    rng = np.random.default_rng(42)
    regimes = [
        "low_entropy_repeating", "sequential_ids", "uniform_32bit",
        "zipfian_smallints", "clustered_ranges"
    ]
    rows = []
    for block, regime in enumerate(regimes * 8):
        for j in range(4):
            base = {
                "low_entropy_repeating": dict(entropy=2.0, rep=0.99, loc=1.0, reuse=0.94, branch=0.02, coh=0.95, pressure=0.02),
                "sequential_ids": dict(entropy=9.0, rep=0.0, loc=1.0, reuse=0.0, branch=0.20, coh=0.55, pressure=0.25),
                "uniform_32bit": dict(entropy=9.0, rep=0.0, loc=0.0, reuse=0.0, branch=0.72, coh=0.08, pressure=0.91),
                "zipfian_smallints": dict(entropy=0.5, rep=0.80, loc=0.27, reuse=0.35, branch=0.72, coh=0.42, pressure=0.78),
                "clustered_ranges": dict(entropy=3.4, rep=0.98, loc=0.0, reuse=0.01, branch=0.79, coh=0.18, pressure=0.99),
            }[regime]
            rows.append({
                "window_id": len(rows),
                "truth_regime": regime,
                "approx_entropy_bits": base["entropy"] + rng.normal(0, 0.08),
                "repetition_ratio": np.clip(base["rep"] + rng.normal(0, 0.02), 0, 1),
                "locality_small_delta_ratio": np.clip(base["loc"] + rng.normal(0, 0.03), 0, 1),
                "cache_window_reuse_proxy": np.clip(base["reuse"] + rng.normal(0, 0.03), 0, 1),
                "branch_pressure_score": np.clip(base["branch"] + rng.normal(0, 0.03), 0, 1),
                "coherence_score": np.clip(base["coh"] + rng.normal(0, 0.03), 0, 1),
                "hardware_pressure_proxy": np.clip(base["pressure"] + rng.normal(0, 0.03), 0, 1),
            })
    df = pd.DataFrame(rows)

df.head()

## Prepare online classification features

In [ ]:
work = df.copy()

# Ensure required columns exist.
defaults = {
    "approx_entropy_bits": 0.0,
    "repetition_ratio": 0.0,
    "locality_small_delta_ratio": 0.0,
    "cache_window_reuse_proxy": 0.0,
    "branch_pressure_score": 0.0,
    "coherence_score": 0.0,
    "hardware_pressure_proxy": 0.0,
    "delta_abs_mean": 0.0,
}

for col, default in defaults.items():
    if col not in work.columns:
        work[col] = default
    work[col] = pd.to_numeric(work[col], errors="coerce").fillna(default)

if "truth_regime" not in work.columns:
    work["truth_regime"] = "unknown"

def norm01(s):
    s = pd.Series(s).astype(float)
    lo, hi = s.min(), s.max()
    if hi == lo:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - lo) / (hi - lo)

work["entropy_norm"] = norm01(work["approx_entropy_bits"])
work["branch_norm"] = norm01(work["branch_pressure_score"])
work["delta_norm"] = norm01(np.log10(work["delta_abs_mean"] + 1.0)) if "delta_abs_mean" in work.columns else 0.0

feature_cols = [
    "approx_entropy_bits",
    "entropy_norm",
    "repetition_ratio",
    "locality_small_delta_ratio",
    "cache_window_reuse_proxy",
    "branch_pressure_score",
    "branch_norm",
    "coherence_score",
    "hardware_pressure_proxy",
]

X = work[feature_cols]
y = work["truth_regime"].astype(str)

print("Feature matrix:", X.shape)
print("Regimes:", sorted(y.unique()))
X.head()

## Train online regime classifiers

A decision tree provides interpretable boundaries.  
A random forest provides a stronger baseline.

In [ ]:
stratify = y if y.value_counts().min() >= 2 else None

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, work.index,
    test_size=0.30,
    random_state=42,
    stratify=stratify
)

tree = DecisionTreeClassifier(max_depth=4, min_samples_leaf=2, random_state=42)
tree.fit(X_train, y_train)

forest = RandomForestClassifier(n_estimators=120, max_depth=6, random_state=42)
forest.fit(X_train, y_train)

pred_tree = tree.predict(X_test)
pred_forest = forest.predict(X_test)

tree_acc = accuracy_score(y_test, pred_tree)
forest_acc = accuracy_score(y_test, pred_forest)

print("Decision tree accuracy:", tree_acc)
print("Random forest accuracy:", forest_acc)

## Predict regimes over all windows

In [ ]:
work["predicted_regime_tree"] = tree.predict(X)
work["predicted_regime_forest"] = forest.predict(X)
work["predicted_regime"] = work["predicted_regime_forest"]
work["regime_prediction_correct"] = work["predicted_regime"] == work["truth_regime"]

# Confidence proxy: top probability minus second probability.
probs = forest.predict_proba(X)
classes = forest.classes_
sorted_probs = np.sort(probs, axis=1)
work["regime_confidence"] = sorted_probs[:, -1] - sorted_probs[:, -2] if probs.shape[1] > 1 else 1.0
work["regime_probability"] = probs.max(axis=1)

work[["window_id", "truth_regime", "predicted_regime", "regime_prediction_correct", "regime_confidence"]].head()

## Map predicted regime to execution policy

This closes the runtime loop:

```text
local window features → predicted regime → policy recommendation
```

In [ ]:
regime_to_policy = {
    "low_entropy_repeating": "coherent_local",
    "sequential_ids": "hybrid",
    "uniform_32bit": "simd",
    "zipfian_smallints": "hybrid",
    "clustered_ranges": "guarded_fallback",
}

work["policy_from_truth"] = work["truth_regime"].map(regime_to_policy).fillna("hybrid")
work["policy_from_predicted_regime"] = work["predicted_regime"].map(regime_to_policy).fillna("hybrid")
work["policy_correct_from_regime"] = work["policy_from_truth"] == work["policy_from_predicted_regime"]

work[[
    "window_id", "truth_regime", "predicted_regime",
    "policy_from_truth", "policy_from_predicted_regime", "policy_correct_from_regime"
]].head()

## Export online classification table

In [ ]:
csv_path = RESULTS_DIR / "notebook11_online_distribution_classification.csv"
json_path = RESULTS_DIR / "notebook11_online_distribution_classification.json"

work.to_csv(csv_path, index=False)
work.to_json(json_path, orient="records", indent=2)

print("Saved:", csv_path)
print("Saved:", json_path)

## Figure 1 — True vs predicted regime timeline

In [ ]:
fig_path_1 = FIGURES_DIR / "notebook11_true_vs_predicted_regime_timeline.png"

labels = sorted(set(work["truth_regime"]).union(set(work["predicted_regime"])))
lab_to_id = {lab: i for i, lab in enumerate(labels)}

plt.figure(figsize=(12, 5))
plt.step(work["window_id"], work["truth_regime"].map(lab_to_id), where="mid", label="true")
plt.step(work["window_id"], work["predicted_regime"].map(lab_to_id), where="mid", label="predicted", linestyle="--")
plt.yticks(list(lab_to_id.values()), list(lab_to_id.keys()))
plt.xlabel("Window")
plt.ylabel("Regime")
plt.title("Online Distribution Classification: True vs Predicted Regime")
plt.legend()
plt.tight_layout()
plt.savefig(fig_path_1, dpi=160)
plt.show()

print("Saved:", fig_path_1)

## Figure 2 — Regime classification confusion matrix

In [ ]:
fig_path_2 = FIGURES_DIR / "notebook11_regime_confusion_matrix.png"

labels = sorted(y.unique())
cm = confusion_matrix(work["truth_regime"], work["predicted_regime"], labels=labels)

plt.figure(figsize=(8, 6))
plt.imshow(cm, aspect="auto")
plt.xticks(range(len(labels)), labels, rotation=45, ha="right")
plt.yticks(range(len(labels)), labels)
plt.xlabel("Predicted regime")
plt.ylabel("True regime")
plt.title("Online Regime Classification Confusion Matrix")
plt.colorbar(label="Count")
plt.tight_layout()
plt.savefig(fig_path_2, dpi=160)
plt.show()

print("Saved:", fig_path_2)

## Figure 3 — Prediction confidence over time

In [ ]:
fig_path_3 = FIGURES_DIR / "notebook11_prediction_confidence_timeline.png"

plt.figure(figsize=(12, 4))
plt.plot(work["window_id"], work["regime_probability"], label="top probability")
plt.plot(work["window_id"], work["regime_confidence"], label="margin confidence")
plt.xlabel("Window")
plt.ylabel("Confidence")
plt.title("Online Regime Prediction Confidence")
plt.legend()
plt.tight_layout()
plt.savefig(fig_path_3, dpi=160)
plt.show()

print("Saved:", fig_path_3)

## Figure 4 — Feature importance

In [ ]:
fig_path_4 = FIGURES_DIR / "notebook11_feature_importance.png"

importances = pd.DataFrame({
    "feature": feature_cols,
    "importance": forest.feature_importances_,
}).sort_values("importance", ascending=False)

plt.figure(figsize=(10, 5))
plt.bar(importances["feature"], importances["importance"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Importance")
plt.title("Online Distribution Classifier: Feature Importance")
plt.tight_layout()
plt.savefig(fig_path_4, dpi=160)
plt.show()

print("Saved:", fig_path_4)

## Figure 5 — Policy correctness from predicted regime

In [ ]:
fig_path_5 = FIGURES_DIR / "notebook11_policy_correctness_from_regime.png"

policy_summary = (
    work.groupby("truth_regime", as_index=False)
    .agg(policy_correct_rate=("policy_correct_from_regime", "mean"))
    .sort_values("policy_correct_rate")
)

plt.figure(figsize=(9, 5))
plt.bar(policy_summary["truth_regime"], policy_summary["policy_correct_rate"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Policy correctness rate")
plt.title("Execution Policy Correctness from Predicted Regime")
plt.tight_layout()
plt.savefig(fig_path_5, dpi=160)
plt.show()

print("Saved:", fig_path_5)

## Figure 6 — Interpretable regime classifier tree

In [ ]:
fig_path_6 = FIGURES_DIR / "notebook11_decision_tree.png"

plt.figure(figsize=(18, 8))
plot_tree(
    tree,
    feature_names=feature_cols,
    class_names=sorted(y.unique()),
    filled=False,
    rounded=True,
    fontsize=7
)
plt.title("Interpretable Online Distribution Classifier")
plt.tight_layout()
plt.savefig(fig_path_6, dpi=180)
plt.show()

print("Saved:", fig_path_6)

## Lab-report summary

In [ ]:
report_path = REPORTS_DIR / "report_11_online_distribution_classification.md"

summary = {
    "windows": int(len(work)),
    "decision_tree_accuracy": float(tree_acc),
    "random_forest_accuracy": float(forest_acc),
    "online_regime_accuracy": float(work["regime_prediction_correct"].mean()),
    "policy_correctness_from_regime": float(work["policy_correct_from_regime"].mean()),
    "mean_regime_probability": float(work["regime_probability"].mean()),
    "mean_regime_confidence": float(work["regime_confidence"].mean()),
}

regime_policy = pd.crosstab(work["truth_regime"], work["policy_from_predicted_regime"])

lines = [
    "# Report 11 — Online Distribution Classification",
    "",
    "This report classifies streaming integer-distribution regimes before selecting execution policies.",
    "",
    "Constraint view:",
    "> adaptive execution depends on detecting structure before selecting policy.",
    "",
    "## Generated outputs",
    "",
    f"- Metrics CSV: `{csv_path}`",
    f"- Metrics JSON: `{json_path}`",
    f"- Figure: `{fig_path_1}`",
    f"- Figure: `{fig_path_2}`",
    f"- Figure: `{fig_path_3}`",
    f"- Figure: `{fig_path_4}`",
    f"- Figure: `{fig_path_5}`",
    f"- Figure: `{fig_path_6}`",
    "",
    "## Summary",
    "",
    pd.DataFrame([summary]).to_markdown(index=False),
    "",
    "## Regime → predicted policy table",
    "",
    regime_policy.to_markdown(),
    "",
    "## Feature importance",
    "",
    importances.to_markdown(index=False),
    "",
    "## Interpretation",
    "",
    "- Online distribution classification moves adaptive execution one step earlier than policy selection.",
    "- Correct regime detection supports correct execution-path routing.",
    "- Confidence traces show when the selector is stable and when it should hesitate.",
    "- Feature importance checks whether the classifier relies on meaningful structure rather than arbitrary labels.",
    "",
    "## Next step",
    "",
    "Notebook 12 can analyze latency-throughput Pareto frontiers: choose policies under explicit tradeoff constraints.",
]

report_path.write_text("\n".join(lines))
print("Saved:", report_path)

## Optional: download output bundle in Colab

Uncomment the following cell if you are running this notebook in Google Colab and want to download generated outputs.

In [ ]:
# OPTIONAL COLAB DOWNLOAD
#
# EXPORT_NAME = "notebook11_online_distribution_classification_outputs.zip"
# export_path = RML_ROOT / EXPORT_NAME
#
# with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for folder in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
#         for p in folder.glob("notebook11_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#         for p in folder.glob("report_11_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#
# from google.colab import files
# files.download(str(export_path))